In [1]:
import pandas as pd
import os

csv_file_name = "full_gene_coords.csv" 
output_txt_name = "gene_list.txt"
# ===========================================

if not os.path.exists(csv_file_name):
    print(f"❌ 错误：当前目录下没找到 {csv_file_name}！")
    print("请先点击左上角的 ↑ 箭头图标，把你电脑上的 CSV 文件上传上来。")
else:
    # 读取 CSV (自动识别分隔符，无论是逗号还是制表符)
    # dtype=str 保证所有数据都按文本读取，防止染色体变数字
    try:
        df = pd.read_csv(csv_file_name, dtype=str)
        
        # 自动清洗列名 (去除空格，转小写)
        df.columns = df.columns.str.strip().str.lower()
        
        # 确认必要的列是否存在
        required_cols = ['chr', 'start', 'end', 'symbol', 'ensembl']
        
        # 检查是否缺列
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            print(f"❌ CSV缺少必要的列: {missing}")
            print(f"你的列名是: {list(df.columns)}")
            print("请确保列名包含: Chr, Start, End, Symbol, Ensembl")
        else:
            # 数据清洗
            # 1. 去掉 Chr 列里的 'chr' 前缀
            df['chr'] = df['chr'].str.replace('chr', '', case=False)
            
            # 2. 选取这5列，并保存为 txt
            # sep='\t' 表示用 Tab 隔开 (非常安全)
            # index=False (不要行号)
            # header=False (不要表头)
            df[required_cols].to_csv(output_txt_name, sep='\t', index=False, header=False)
            
            print("✅ 转换成功！")
            print(f"文件已保存为: {output_txt_name}")
            print("\n⬇️ 文件预览 (前 5 行):")
            # 预览一下生成的格式对不对
            with open(output_txt_name, 'r') as f:
                print(f.read(300)) # 只打印前300个字符
                
    except Exception as e:
        print(f"❌ 读取或转换失败: {e}")

✅ 转换成功！
文件已保存为: gene_list.txt

⬇️ 文件预览 (前 5 行):
12	110124335	110218795	IFT81	ENSG00000122970
9	26947039	27066134	IFT74	ENSG00000096872
3	93980139	94055678	ARL13B	ENSG00000169379
X	38269163	38327544	RPGR	ENSG00000156313
16	90019629	90044975	GAS8	ENSG00000141013
2	61824848	61854143	FAM161A	ENSG00000170264
2	84516455	84819589	DNAH6	ENSG00000115423
1


In [4]:
import os

# ================= 配置 =================
# 原始的那个可能有问题的 PSAM 路径
input_psam = "/mnt/project/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c1_b0_v1.psam"
# 我们将要生成的修复版文件
output_psam = "./fixed_samples.psam"
# =======================================

print(f"🔧 正在尝试修复样本文件: {os.path.basename(input_psam)}")

try:
    valid_lines = []
    
    # 1. 使用二进制模式读取，避开编码报错
    with open(input_psam, 'rb') as f:
        raw_content = f.read()
        
    # 检查是否全是空字节 (云端挂载常见问题)
    if all(b == 0 for b in raw_content[:1024]):
        print("❌ 致命诊断：这个文件全是 '空字节' (Null Bytes)！")
        print("💡 解决办法：这是云端挂载的问题。请去 UKB-RAP 网页端，找到这个文件夹，双击点开以此来'唤醒'文件，或者稍等几分钟再试。")
        # 这种情况下我们无法修复，只能停止
    else:
        # 2. 尝试解码并提取文本
        text_content = raw_content.decode('utf-8', errors='ignore')
        lines = text_content.strip().split('\n')
        
        print(f"   读取到 {len(lines)} 行原始数据...")
        
        # 3. 检查是否有表头
        has_header = False
        if lines and (lines[0].startswith('#') or 'ID' in lines[0]):
            print("   (看起来原来就有表头)")
            has_header = True
            
        # 4. 写入新文件，强制加上标准表头
        with open(output_psam, 'w') as out:
            # PLINK2 通常需要 #IID 或者 #FID IID
            # 我们先看第一行数据长什么样
            first_data_line = lines[1] if has_header and len(lines)>1 else lines[0]
            cols = first_data_line.split()
            
            # 根据列数决定表头
            if len(cols) == 1:
                out.write("#IID\n") # 只有一列 ID
            elif len(cols) >= 2:
                out.write("#FID\tIID\n") # 两列 (通常是 FamilyID SampleID)
                
            # 写入数据
            count = 0
            for line in lines:
                if line.startswith('#') or 'ID' in line: continue # 跳过旧表头
                clean_line = line.strip()
                if clean_line:
                    # 将空格统一转换为制表符，确保格式完美
                    parts = clean_line.split()
                    out.write("\t".join(parts) + "\n")
                    count += 1
                    
        print(f"✅ 修复成功！已生成新文件: {output_psam}")
        print(f"   包含样本数: {count}")
        print("   现在可以用这个文件去跑提取了！")

except Exception as e:
    print(f"❌ 修复过程中出错: {e}")

🔧 正在尝试修复样本文件: ukb24308_c1_b0_v1.psam
   读取到 490541 行原始数据...
✅ 修复成功！已生成新文件: ./fixed_samples.psam
   包含样本数: 490541
   现在可以用这个文件去跑提取了！
